# MODEL 8 — AFI / AMNIOTIC FLUID LONGITUDINAL ENGINE
### PregnancyTwin AI — 4-Quadrant AFI & DVP Fluid Volume Dynamics, First & Second Order Longitudinal Derivatives, Linear Trend Slope & Data-Quality Auditing

```text
                     ULTRASOUND IMAGE
                            │
                            ▼
               ┌────────────────────────┐
               │ MODEL 1                │
               │ Image Quality Gate AI  │ ➔ [SNR, Sharpness, Acoustic Shadow Gate]
               └────────────┬───────────┘
                            │
                            ▼
               ┌────────────────────────┐
               │ MODEL 2                │
               │ View Classification AI │ ➔ [Swin-T: HEAD, ABDOMEN, FEMUR]
               └────────────┬───────────┘
                            │
          ┌─────────────────┼─────────────────┐
          ▼                 ▼                 ▼
        HEAD             ABDOMEN            FEMUR
          │                 │                 │
          ▼                 ▼                 ▼
       MODEL 3           MODEL 4           MODEL 5
      Head U-Net      Abdomen U-Net      Femur U-Net
          │                 │                 │
          ▼                 ▼                 ▼
      Head Mask        Abdomen Mask      Femur Mask
          │                 │                 │
          └─────────────────┼─────────────────┘
                            ▼
               ┌────────────────────────┐
               │ MODEL 6                │
               │ BIOMETRY & CALIBRATION │ ➔ [HC, BPD, OFD, AC, FL in mm]
               └────────────┬───────────┘
                            │
                            ▼
               ┌────────────────────────┐
               │ MODEL 7                │
               │ EFW & GROWTH ENGINE    │ ➔ [EFW + Fetal Growth Trajectory]
               └────────────┬───────────┘
                            │
         ┌──────────────────┴──────────────────┐
         │                                     │
         ▼                                     ▼
  GROWTH TRAJECTORY                   MODEL 8: AMNIOTIC FLUID
  (EFW, HC, AC, FL)                   (AFI / DVP Longitudinal Engine)
         │                                     │
         └──────────────────┬──────────────────┘
                            ▼
               ┌────────────────────────┐
               │ PREGNANCY DIGITAL TWIN │ ➔ [Multi-Modal Longitudinal Fusion]
               └────────────┬───────────┘
                            ▼
               ┌────────────────────────┐
               │ XGBoost + Isolation    │ ➔ [Predictive Risk & Anomaly Signals]
               │ Forest + SHAP Explain  │
               └────────────────────────┘
```

**Core Clinical Principle**: *Do not treat AFI as a single isolated number. Model how amniotic-fluid volume, velocity, acceleration, and trend slope evolve across gestation.*

In [ ]:
# SECTION 1 — Imports & Library Installation
!pip install -q numpy scipy pandas matplotlib seaborn scikit-learn

import math, json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

print("✓ Model 8 Environment Initialized. NumPy:", np.__version__, "Pandas:", pd.__version__)

In [ ]:
# SECTION 2 — Master Fluid Configuration & Reference Standards
FLUID_CONFIG = {
    "engine_name": "PregnancyTwin Amniotic Fluid Longitudinal Engine",
    "engine_version": "v8.0-production",
    "afi_reference_standard": "Moore & Cayle 4-Quadrant AFI Norms (1990)",
    "dvp_reference_standard": "Manning Biophysical Profile DVP Standard (1980)",
    "normal_afi_range_cm": [8.0, 24.0],
    "borderline_low_afi_cm": [5.0, 8.0],
    "oligohydramnios_afi_threshold_cm": 5.0,
    "polyhydramnios_afi_threshold_cm": 24.0,
    "normal_dvp_range_cm": [2.0, 8.0],
    "rolling_window_scans": 3
}
print("Master Fluid Configuration:", json.dumps(FLUID_CONFIG, indent=2))

In [ ]:
# SECTION 3 — Load Longitudinal Pregnancy Dataset with Fluid Measurements
# Simulating multi-visit clinical record for patient PT-001 (GA 24w, 28w, 32w, 36w)
raw_pregnancy_visits = [
    {
        "visit_id": "V1",
        "visit_date": "2026-05-15",
        "gestational_age_weeks": 24.0,
        "gestational_age_days": 168,
        "q1_cm": 3.8,
        "q2_cm": 4.1,
        "q3_cm": 3.2,
        "q4_cm": 3.4,
        "afi_cm": 14.5,
        "dvp_cm": 4.8,
        "ultrasound_quality": 0.94,
        "measurement_confidence": 0.96,
        "doppler_available": False # Preserved as false; no artificial fabrication
    },
    {
        "visit_id": "V2",
        "visit_date": "2026-06-12",
        "gestational_age_weeks": 28.0,
        "gestational_age_days": 196,
        "q1_cm": 3.2,
        "q2_cm": 3.5,
        "q3_cm": 2.8,
        "q4_cm": 3.0,
        "afi_cm": 12.5,
        "dvp_cm": 4.2,
        "ultrasound_quality": 0.91,
        "measurement_confidence": 0.93,
        "doppler_available": False
    },
    {
        "visit_id": "V3",
        "visit_date": "2026-07-10",
        "gestational_age_weeks": 32.0,
        "gestational_age_days": 224,
        "q1_cm": 2.4,
        "q2_cm": 2.8,
        "q3_cm": 2.1,
        "q4_cm": 2.5,
        "afi_cm": 9.8,
        "dvp_cm": 3.5,
        "ultrasound_quality": 0.88,
        "measurement_confidence": 0.90,
        "doppler_available": False
    },
    {
        "visit_id": "V4",
        "visit_date": "2026-08-07",
        "gestational_age_weeks": 36.0,
        "gestational_age_days": 252,
        "q1_cm": 1.9,
        "q2_cm": 2.2,
        "q3_cm": 1.8,
        "q4_cm": 2.1,
        "afi_cm": 8.0,
        "dvp_cm": 2.9,
        "ultrasound_quality": 0.86,
        "measurement_confidence": 0.89,
        "doppler_available": False
    }
]
print(f"Loaded {len(raw_pregnancy_visits)} fluid visit records.")

In [ ]:
# SECTION 4 & 5 — Validate AFI / DVP & Handle Missing Values (None vs 0)
def validate_fluid_record(visit):
    # Never replace missing with 0; preserve None
    afi = visit.get("afi_cm")
    dvp = visit.get("dvp_cm")
    
    # Verify 4-quadrant sum if present
    qs = [visit.get(f"q{i}_cm") for i in range(1, 5)]
    if all(q is not None for q in qs):
        q_sum = round(sum(qs), 1)
        if afi is not None and abs(afi - q_sum) > 0.3:
            print(f"Notice: Visit {visit['visit_id']} AFI ({afi}cm) differs slightly from quadrant sum ({q_sum}cm)")
            
    return {
        "afi_valid": (afi is not None and 0.0 <= afi <= 45.0),
        "dvp_valid": (dvp is not None and 0.0 <= dvp <= 20.0),
        "afi_cm": round(afi, 1) if afi is not None else None,
        "dvp_cm": round(dvp, 1) if dvp is not None else None
    }

for v in raw_pregnancy_visits:
    res = validate_fluid_record(v)
    print(f"Visit {v['visit_id']}: AFI={res['afi_cm']} cm, DVP={res['dvp_cm']} cm | Valid: {res['afi_valid']}")

In [ ]:
# SECTION 6 & 7 — Chronological Sorting & Interval Time Gap Calculation
# Step 1: Ensure dataset is strictly sorted by gestational age / date
sorted_visits = sorted(raw_pregnancy_visits, key=lambda x: x["gestational_age_days"])

for i in range(len(sorted_visits)):
    curr = sorted_visits[i]
    if i == 0:
        curr["time_gap_days"] = 0
        curr["time_gap_weeks"] = 0.0
    else:
        prev = sorted_visits[i-1]
        gap = curr["gestational_age_days"] - prev["gestational_age_days"]
        curr["time_gap_days"] = gap
        curr["time_gap_weeks"] = round(gap / 7.0, 2)
        
    print(f"Visit {curr['visit_id']} (GA {curr['gestational_age_weeks']}w): Time gap from prev = {curr['time_gap_days']} days ({curr['time_gap_weeks']} weeks)")

In [ ]:
# SECTION 8, 9, 10 — Previous AFI, AFI Delta (ΔAFI), and Percentage Change
for i in range(len(sorted_visits)):
    curr = sorted_visits[i]
    if i == 0:
        curr["previous_afi_cm"] = None
        curr["afi_delta_cm"] = None
        curr["afi_percent_change"] = None
    else:
        prev = sorted_visits[i-1]
        curr["previous_afi_cm"] = prev["afi_cm"]
        if curr["afi_cm"] is not None and prev["afi_cm"] is not None:
            delta = round(curr["afi_cm"] - prev["afi_cm"], 1)
            pct = round((delta / prev["afi_cm"]) * 100.0, 1)
            curr["afi_delta_cm"] = delta
            curr["afi_percent_change"] = pct
            print(f"Visit {curr['visit_id']}: AFI {prev['afi_cm']} -> {curr['afi_cm']} cm | ΔAFI = {delta} cm ({pct}%)")
        else:
            curr["afi_delta_cm"] = None
            curr["afi_percent_change"] = None

In [ ]:
# SECTION 11 & 12 — AFI Velocity (cm/day & cm/week) and Second-Order Acceleration
for i in range(len(sorted_visits)):
    curr = sorted_visits[i]
    if i == 0 or curr["afi_delta_cm"] is None or curr["time_gap_days"] <= 0:
        curr["afi_velocity_cm_per_day"] = None
        curr["afi_velocity_cm_per_week"] = None
        curr["afi_acceleration"] = None
    else:
        dt_days = curr["time_gap_days"]
        dt_weeks = curr["time_gap_weeks"]
        vel_day = round(curr["afi_delta_cm"] / dt_days, 4)
        vel_week = round(curr["afi_delta_cm"] / dt_weeks, 2)
        curr["afi_velocity_cm_per_day"] = vel_day
        curr["afi_velocity_cm_per_week"] = vel_week
        
        # Acceleration (2nd order derivative)
        if i >= 2 and sorted_visits[i-1]["afi_velocity_cm_per_week"] is not None:
            prev_vel = sorted_visits[i-1]["afi_velocity_cm_per_week"]
            accel = round((vel_week - prev_vel) / dt_weeks, 2)
            curr["afi_acceleration"] = accel
        else:
            curr["afi_acceleration"] = None
            
        print(f"Visit {curr['visit_id']}: Velocity = {vel_week} cm/week ({vel_day} cm/day) | Acceleration = {curr['afi_acceleration']} cm/wk²")

In [ ]:
# SECTION 13 & 14 — Multi-Scan Rolling Features & Linear Trend Slope
for i in range(len(sorted_visits)):
    window = sorted_visits[max(0, i-2):i+1]
    afis = [v["afi_cm"] for v in window if v.get("afi_cm") is not None]
    sorted_visits[i]["afi_rolling_mean"] = round(np.mean(afis), 1) if afis else None
    sorted_visits[i]["afi_rolling_median"] = round(float(np.median(afis)), 1) if afis else None

# Compute linear regression slope (cm / week) across all available scans
valid_gas = [v["gestational_age_weeks"] for v in sorted_visits if v.get("afi_cm") is not None]
valid_afis = [v["afi_cm"] for v in sorted_visits if v.get("afi_cm") is not None]

if len(valid_gas) >= 2:
    slope, intercept = np.polyfit(valid_gas, valid_afis, 1)
    slope = round(slope, 3)
else:
    slope = 0.0

for v in sorted_visits:
    v["afi_trend_slope"] = slope
    print(f"Visit {v['visit_id']}: Rolling Mean AFI = {v['afi_rolling_mean']} cm | Multi-Scan Trend Slope = {slope} cm/week")

In [ ]:
# SECTION 15 — Deepest Vertical Pocket (DVP) Longitudinal Features
for i in range(len(sorted_visits)):
    curr = sorted_visits[i]
    if i == 0:
        curr["previous_dvp_cm"] = None
        curr["dvp_delta_cm"] = None
        curr["dvp_velocity"] = None
    else:
        prev = sorted_visits[i-1]
        curr["previous_dvp_cm"] = prev.get("dvp_cm")
        if curr.get("dvp_cm") is not None and prev.get("dvp_cm") is not None:
            d_delta = round(curr["dvp_cm"] - prev["dvp_cm"], 1)
            curr["dvp_delta_cm"] = d_delta
            curr["dvp_velocity"] = round(d_delta / curr["time_gap_weeks"], 2)
            print(f"Visit {curr['visit_id']}: DVP {prev['dvp_cm']} -> {curr['dvp_cm']} cm | ΔDVP = {d_delta} cm (Vel: {curr['dvp_velocity']} cm/wk)")
        else:
            curr["dvp_delta_cm"] = None
            curr["dvp_velocity"] = None

In [ ]:
# SECTION 16 — Consecutive Declining Scans & Trajectory Direction Classification
for i in range(len(sorted_visits)):
    declines = 0
    for k in range(i, 0, -1):
        if (sorted_visits[k].get("afi_cm") is not None and 
            sorted_visits[k-1].get("afi_cm") is not None and 
            sorted_visits[k]["afi_cm"] < sorted_visits[k-1]["afi_cm"] - 0.5):
            declines += 1
        else:
            break
            
    sorted_visits[i]["consecutive_declining_afi_visits"] = declines
    
    # Categorize trajectory
    delta = sorted_visits[i].get("afi_delta_cm") or 0.0
    if declines >= 2 or delta < -2.0:
        direction = "DECLINING"
    elif delta > 2.0:
        direction = "INCREASING"
    else:
        direction = "STABLE"
        
    sorted_visits[i]["fluid_trajectory_direction"] = direction
    print(f"Visit {sorted_visits[i]['visit_id']}: Consecutive Declining Scans = {declines} -> Trajectory: {direction}")

In [ ]:
# SECTION 17 — Fluid Data Quality Auditing (GOOD / PARTIAL / REVIEW / INSUFFICIENT)
for v in sorted_visits:
    has_afi = v.get("afi_cm") is not None
    has_dvp = v.get("dvp_cm") is not None
    conf = v.get("measurement_confidence", 0.90)
    
    if not has_afi and not has_dvp:
        qual = "INSUFFICIENT"
    elif conf < 0.70:
        qual = "REVIEW"
    elif has_afi and has_dvp:
        qual = "GOOD"
    else:
        qual = "PARTIAL"
        
    v["fluid_data_quality"] = qual
    print(f"Visit {v['visit_id']}: Quality Status = {qual} (Confidence: {conf*100:.1f}%)")

In [ ]:
# SECTION 18 — Longitudinal Amniotic Fluid Trajectory Visualization
plt.figure(figsize=(10, 5.5))

# Normal AFI Reference Range (8 - 24 cm)
ga_plot = np.linspace(20, 40, 100)
plt.axhspan(8.0, 24.0, color="teal", alpha=0.12, label="Normal AFI Range (8.0 - 24.0 cm)")
plt.axhspan(5.0, 8.0, color="amber", alpha=0.15, label="Borderline Low (5.0 - 8.0 cm)")
plt.axhline(5.0, color="crimson", linestyle="--", linewidth=1.5, label="Oligohydramnios Threshold (<5.0 cm)")

# Patient Scans
gas = [v["gestational_age_weeks"] for v in sorted_visits]
pt_afis = [v["afi_cm"] for v in sorted_visits]
plt.plot(gas, pt_afis, "-o", color="#0284c7", linewidth=2.5, markersize=8, label="Patient AFI Trajectory (PT-001)")

# Trendline
trend_y = [valid_afis[0] + slope * (g - valid_gas[0]) for g in gas]
plt.plot(gas, trend_y, ":", color="#64748b", linewidth=1.8, label=f"Fitted Linear Trend (Slope: {slope} cm/wk)")

for v in sorted_visits:
    plt.annotate(f"{v['afi_cm']} cm", (v["gestational_age_weeks"], v["afi_cm"]),
                 textcoords="offset points", xytext=(0,10), ha="center", fontweight="bold")

plt.title("MODEL 8: Longitudinal Amniotic Fluid Index (AFI) Trajectory & Trendline", fontsize=13, fontweight="bold")
plt.xlabel("Gestational Age (Weeks)", fontweight="bold")
plt.ylabel("AFI (cm)", fontweight="bold")
plt.ylim(0, 30)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# SECTION 19 — Export Serialized Longitudinal Fluid Feature Table
df_fluid = pd.DataFrame(sorted_visits)
export_fluid_cols = [
    "visit_id", "gestational_age_weeks", "afi_cm", "dvp_cm", "time_gap_days",
    "previous_afi_cm", "afi_delta_cm", "afi_percent_change",
    "afi_velocity_cm_per_week", "afi_acceleration", "afi_rolling_mean",
    "afi_trend_slope", "consecutive_declining_afi_visits", "fluid_trajectory_direction", "fluid_data_quality"
]
print("=== SERIALIZED MODEL 8 FLUID FEATURE MATRIX ===")
print(df_fluid[export_fluid_cols].to_string(index=False))

df_fluid[export_fluid_cols].to_csv("longitudinal_amniotic_fluid_features.csv", index=False)
print("✓ Exported longitudinal_amniotic_fluid_features.csv for downstream Digital Twin fusion.")

In [ ]:
# SECTION 20 — Multimodal Fusion with Model 7 Growth Trajectory
print("=== MODEL 7 (GROWTH) + MODEL 8 (FLUID) MULTIMODAL FEATURE FUSION ===")
fused_vector_example = {
    "gestational_age_weeks": 36.0,
    "EFW_g": 2480.0,
    "EFW_velocity_g_per_week": 165.0,
    "EFW_acceleration": -12.0,
    "growth_percentile": 38.5,
    "AFI_cm": 8.0,
    "AFI_velocity_cm_per_week": -0.45,
    "AFI_trend_slope": -0.54,
    "DVP_cm": 2.9,
    "consecutive_declining_afi_visits": 3,
    "fluid_trajectory_state": "DECLINING",
    "time_gap_days": 28
}
print(json.dumps(fused_vector_example, indent=2))
print("✓ Ready for Model 9 Maternal Context & Model 10 XGBoost / Isolation Forest Architecture.")